In [3]:
"""
Task 2: Customer Segmentation Using Unsupervised Learning
=========================================================
Dataset: Mall Customers (generated to match real schema)
Methods: K-Means Clustering, PCA, t-SNE
Output:  Cluster profiles + Marketing strategies
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, silhouette_samples
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ─────────────────────────────────────────────
# 1. GENERATE MALL CUSTOMERS DATASET
# ─────────────────────────────────────────────
def generate_mall_customers(n=200):
    """Synthetic dataset matching Mall Customers schema with 5 natural clusters."""
    segments = [
        # (age_mu, age_sd, income_mu, income_sd, score_mu, score_sd, n_share)
        (25, 5,  30, 8,  75, 12, 0.20),   # Young low-income high-spenders
        (42, 8,  75, 12, 65, 10, 0.20),   # Middle-age high-income high-spenders
        (40, 9,  55, 10, 48, 10, 0.20),   # Middle-age average
        (55, 8,  80, 15, 20, 10, 0.20),   # Older high-income low-spenders
        (30, 6,  25, 7,  20,  9, 0.20),   # Young low-income low-spenders
    ]
    rows = []
    cid  = 1
    for seg_idx, (am, asd, im, isd, sm, ssd, share) in enumerate(segments):
        k = int(n * share)
        for _ in range(k):
            gender  = np.random.choice(['Male','Female'])
            age     = int(np.clip(np.random.normal(am, asd), 18, 70))
            income  = int(np.clip(np.random.normal(im, isd), 15, 137))
            score   = int(np.clip(np.random.normal(sm, ssd), 1, 99))
            rows.append({'CustomerID': cid, 'Gender': gender,
                         'Age': age, 'Annual Income (k$)': income,
                         'Spending Score (1-100)': score})
            cid += 1
    return pd.DataFrame(rows)

df = generate_mall_customers(200)
print(f"Dataset shape: {df.shape}")
print(df.describe().round(2))

# ─────────────────────────────────────────────
# 2. EDA FIGURE
# ─────────────────────────────────────────────
fig_eda, axes = plt.subplots(2, 3, figsize=(18, 10), facecolor='#0f1117')
fig_eda.suptitle('Task 2: Mall Customers — Exploratory Data Analysis',
                 fontsize=16, fontweight='bold', color='white', y=0.98)

eda_vars = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
palette  = {'Male': '#4FC3F7', 'Female': '#F48FB1'}

# Row 0: Distributions
for i, var in enumerate(eda_vars):
    ax = axes[0, i]
    ax.set_facecolor('#1a1d27')
    for gender, grp in df.groupby('Gender'):
        ax.hist(grp[var], bins=20, alpha=0.7, color=palette[gender], label=gender, edgecolor='#0f1117')
    ax.set_title(f'Distribution of {var}', color='white', fontsize=11)
    ax.set_xlabel(var, color='white')
    ax.set_ylabel('Count', color='white')
    ax.legend(facecolor='#252836', labelcolor='white', fontsize=9)
    ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_edgecolor('#333')
    ax.grid(alpha=0.1, color='white')

# Row 1: Scatter plots
scatter_pairs = [
    ('Age', 'Annual Income (k$)'),
    ('Annual Income (k$)', 'Spending Score (1-100)'),
    ('Age', 'Spending Score (1-100)'),
]
for i, (xvar, yvar) in enumerate(scatter_pairs):
    ax = axes[1, i]
    ax.set_facecolor('#1a1d27')
    for gender, grp in df.groupby('Gender'):
        ax.scatter(grp[xvar], grp[yvar], alpha=0.7, s=40,
                   color=palette[gender], label=gender, edgecolors='none')
    ax.set_xlabel(xvar, color='white')
    ax.set_ylabel(yvar, color='white')
    ax.set_title(f'{xvar} vs {yvar}', color='white', fontsize=10)
    ax.legend(facecolor='#252836', labelcolor='white', fontsize=9)
    ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_edgecolor('#333')
    ax.grid(alpha=0.1, color='white')

plt.tight_layout()
plt.savefig('./task2_eda.png', dpi=150,
            bbox_inches='tight', facecolor='#0f1117')
plt.close()
print(" Saved: task2_eda.png")

# ─────────────────────────────────────────────
# 3. OPTIMAL K SELECTION
# ─────────────────────────────────────────────
features = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X = df[features].values
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

inertias    = []
sil_scores  = []
K_range     = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=20, random_state=42)
    km.fit(X_sc)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_sc, km.labels_))

best_k = K_range[np.argmax(sil_scores)]
print(f"\nOptimal K by Silhouette: {best_k}")

# ─────────────────────────────────────────────
# 4. FIT FINAL K-MEANS
# ─────────────────────────────────────────────
km_final = KMeans(n_clusters=5, init='k-means++', n_init=20, random_state=42)
df['Cluster'] = km_final.fit_predict(X_sc)
sil_final = silhouette_score(X_sc, df['Cluster'])
print(f"Final K=5 Silhouette Score: {sil_final:.4f}")

# Cluster profile
profile = df.groupby('Cluster')[features].mean().round(1)
profile['Count'] = df['Cluster'].value_counts().sort_index()
print(f"\nCluster Profiles:\n{profile}")

# ─────────────────────────────────────────────
# 5. PCA & t-SNE REDUCTION
# ─────────────────────────────────────────────
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sc)

tsne   = TSNE(n_components=2, perplexity=30, max_iter=1000, random_state=42)
X_tsne = tsne.fit_transform(X_sc)

# ─────────────────────────────────────────────
# 6. MAIN CLUSTERING FIGURE
# ─────────────────────────────────────────────
cluster_colors = ['#FF6B6B','#4ECDC4','#FFE66D','#A29BFE','#55EFC4']
cluster_labels  = {
    0: 'Budget Shoppers',
    1: 'High Earner\nHigh Spenders',
    2: 'Standard\nCustomers',
    3: 'Wealthy\nConservatives',
    4: 'Young\nSpendthrifts',
}

fig2 = plt.figure(figsize=(22, 18), facecolor='#0f1117')
fig2.suptitle('Task 2: Customer Segmentation — K-Means Clustering Analysis',
              fontsize=17, fontweight='bold', color='white', y=0.99)

gs = gridspec.GridSpec(3, 3, figure=fig2, hspace=0.42, wspace=0.35)

# ── Elbow + Silhouette ──
ax_elbow = fig2.add_subplot(gs[0, 0])
ax_elbow.set_facecolor('#1a1d27')
ax_elbow.plot(list(K_range), inertias, 'o-', color='#4FC3F7', lw=2)
ax_elbow.axvline(5, color='#FFB74D', ls='--', lw=1.5, label='K=5 chosen')
ax_elbow.set_title('Elbow Method (Inertia)', color='white', fontsize=11)
ax_elbow.set_xlabel('Number of Clusters K', color='white')
ax_elbow.set_ylabel('Inertia', color='white')
ax_elbow.legend(facecolor='#252836', labelcolor='white')
ax_elbow.tick_params(colors='white')
for sp in ax_elbow.spines.values(): sp.set_edgecolor('#333')
ax_elbow.grid(alpha=0.12, color='white')

ax_sil = fig2.add_subplot(gs[0, 1])
ax_sil.set_facecolor('#1a1d27')
ax_sil.plot(list(K_range), sil_scores, 's-', color='#81C784', lw=2)
ax_sil.axvline(5, color='#FFB74D', ls='--', lw=1.5, label='K=5 chosen')
ax_sil.set_title('Silhouette Score vs K', color='white', fontsize=11)
ax_sil.set_xlabel('Number of Clusters K', color='white')
ax_sil.set_ylabel('Silhouette Score', color='white')
ax_sil.legend(facecolor='#252836', labelcolor='white')
ax_sil.tick_params(colors='white')
for sp in ax_sil.spines.values(): sp.set_edgecolor('#333')
ax_sil.grid(alpha=0.12, color='white')

# ── Original Space: Income vs Spending ──
ax_scatter = fig2.add_subplot(gs[0, 2])
ax_scatter.set_facecolor('#1a1d27')
for c in range(5):
    mask = df['Cluster'] == c
    ax_scatter.scatter(df.loc[mask, 'Annual Income (k$)'],
                       df.loc[mask, 'Spending Score (1-100)'],
                       c=cluster_colors[c], s=60, alpha=0.85,
                       label=f'C{c}', edgecolors='none')
centroids_orig = scaler.inverse_transform(km_final.cluster_centers_)
ax_scatter.scatter(centroids_orig[:, 1], centroids_orig[:, 2],
                   c='white', s=200, marker='X', zorder=5, label='Centroids')
ax_scatter.set_title('Income vs Spending\n(Original Space)', color='white', fontsize=11)
ax_scatter.set_xlabel('Annual Income (k$)', color='white')
ax_scatter.set_ylabel('Spending Score', color='white')
ax_scatter.legend(facecolor='#252836', labelcolor='white', fontsize=8, ncol=3)
ax_scatter.tick_params(colors='white')
for sp in ax_scatter.spines.values(): sp.set_edgecolor('#333')
ax_scatter.grid(alpha=0.1, color='white')

# ── PCA Plot ──
ax_pca = fig2.add_subplot(gs[1, 0])
ax_pca.set_facecolor('#1a1d27')
for c in range(5):
    mask = df['Cluster'] == c
    ax_pca.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=cluster_colors[c], s=55, alpha=0.85, label=f'C{c}', edgecolors='none')
ax_pca.set_title(f'PCA Visualization\n(Explained Var: {pca.explained_variance_ratio_.sum():.1%})',
                 color='white', fontsize=11)
ax_pca.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', color='white')
ax_pca.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', color='white')
ax_pca.tick_params(colors='white')
for sp in ax_pca.spines.values(): sp.set_edgecolor('#333')
ax_pca.grid(alpha=0.1, color='white')

# ── t-SNE Plot ──
ax_tsne = fig2.add_subplot(gs[1, 1])
ax_tsne.set_facecolor('#1a1d27')
for c in range(5):
    mask = df['Cluster'] == c
    ax_tsne.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    c=cluster_colors[c], s=55, alpha=0.85, label=f'C{c}', edgecolors='none')
ax_tsne.set_title('t-SNE Visualization\n(Perplexity=30)', color='white', fontsize=11)
ax_tsne.set_xlabel('t-SNE 1', color='white')
ax_tsne.set_ylabel('t-SNE 2', color='white')
ax_tsne.tick_params(colors='white')
for sp in ax_tsne.spines.values(): sp.set_edgecolor('#333')
ax_tsne.grid(alpha=0.1, color='white')

# ── Cluster Profile Radar ──
ax_profile = fig2.add_subplot(gs[1, 2])
ax_profile.set_facecolor('#1a1d27')
profile_norm = (profile[features] - profile[features].min()) / (profile[features].max() - profile[features].min())
x_pos = np.arange(len(features))
for c in range(5):
    ax_profile.plot(x_pos, profile_norm.loc[c].values, 'o-',
                    color=cluster_colors[c], label=f'C{c}', lw=2, markersize=6)
ax_profile.set_xticks(x_pos)
ax_profile.set_xticklabels(['Age','Income','Score'], color='white', fontsize=9)
ax_profile.set_title('Cluster Profiles\n(Normalized Features)', color='white', fontsize=11)
ax_profile.set_ylabel('Normalized Value', color='white')
ax_profile.legend(facecolor='#252836', labelcolor='white', fontsize=8, ncol=5,
                  loc='upper center', bbox_to_anchor=(0.5, -0.12))
ax_profile.tick_params(colors='white')
for sp in ax_profile.spines.values(): sp.set_edgecolor('#333')
ax_profile.grid(alpha=0.12, color='white')

# ── Marketing Strategies (text panel spanning full width) ──
ax_strat = fig2.add_subplot(gs[2, :])
ax_strat.set_facecolor('#1a1d27')
ax_strat.axis('off')

strategies = [
    ('C0 – Budget Shoppers\n(Low Income, Low Score)',
     '• Launch loyalty reward programs with cashback incentives\n'
     '• Promote budget-friendly bundles & BOGO offers\n'
     '• Retarget with discount email campaigns',
     cluster_colors[0]),
    ('C1 – High Earner High Spenders\n(High Income, High Score)',
     '• Offer VIP/premium membership tiers\n'
     '• Introduce exclusive early-access & luxury products\n'
     '• Personal stylist & concierge upsell programs',
     cluster_colors[1]),
    ('C2 – Standard Customers\n(Mid Income, Mid Score)',
     '• Cross-sell complementary products\n'
     '• Seasonal promotions & limited-time flash sales\n'
     '• Referral bonuses to increase engagement',
     cluster_colors[2]),
    ('C3 – Wealthy Conservatives\n(High Income, Low Score)',
     '• Target with quality-over-quantity messaging\n'
     '• Provide personalized needs-based recommendations\n'
     '• Highlight warranties, premium service & trust signals',
     cluster_colors[3]),
    ('C4 – Young Spendthrifts\n(Low Income, High Score)',
     '• Leverage social media & influencer campaigns\n'
     '• Offer "buy now pay later" flexible payment options\n'
     '• Gamify shopping experience with challenges',
     cluster_colors[4]),
]

ax_strat.set_title('Marketing Strategies by Customer Segment',
                   color='white', fontsize=13, fontweight='bold', pad=10, loc='left')

for i, (title, body, color) in enumerate(strategies):
    x_start = i * 0.205
    rect = FancyBboxPatch((x_start, 0.05), 0.195, 0.85,
                          boxstyle="round,pad=0.015", linewidth=2,
                          edgecolor=color, facecolor='#252836',
                          transform=ax_strat.transAxes)
    ax_strat.add_patch(rect)
    ax_strat.text(x_start + 0.097, 0.82, title,
                  transform=ax_strat.transAxes,
                  ha='center', va='top', fontsize=8, fontweight='bold',
                  color=color, wrap=True)
    ax_strat.text(x_start + 0.097, 0.57, body,
                  transform=ax_strat.transAxes,
                  ha='center', va='top', fontsize=7.5, color='#ddd',
                  linespacing=1.6)

plt.savefig('./task2_clustering.png', dpi=150,
            bbox_inches='tight', facecolor='#0f1117')
plt.close()
print(" Saved: task2_clustering.png")
print("\n Task 2 complete.")

Dataset shape: (200, 5)
       CustomerID     Age  Annual Income (k$)  Spending Score (1-100)
count      200.00  200.00              200.00                  200.00
mean       100.50   38.29               53.04                   46.46
std         57.88   12.65               25.68                   25.90
min          1.00   18.00               15.00                    1.00
25%         50.75   28.00               30.00                   22.00
50%        100.50   36.50               55.00                   48.50
75%        150.25   48.00               72.25                   69.00
max        200.00   70.00              118.00                   99.00
 Saved: task2_eda.png

Optimal K by Silhouette: 4
Final K=5 Silhouette Score: 0.3961

Cluster Profiles:
          Age  Annual Income (k$)  Spending Score (1-100)  Count
Cluster                                                         
0        55.4                83.2                    21.2     39
1        30.7                24.9              